# SolarSDE Part 1 — Foundations (~9-11 hours)

**Run this notebook FIRST.** Produces the Golden CO VAE + latents + Stanford SKIPP'D pipeline.

## Stages

| Stage | What it does | Est. runtime |
|-------|--------------|--------------|
| Retrain (conditional) | Download CloudCV + BMS, preprocess, train Golden VAE, extract latents, physics features, extended splits | ~3h (if nothing cached) |
| Stage A | Stanford SKIPP'D: download, train VAE, extract latents, train SDE + Score Decoder, evaluate | ~6-8h |
| Zip | Package all outputs under `/kaggle/working/solarsde_outputs/` | <5 min |

## Kaggle workflow

1. Enable P100 GPU + Internet (Settings panel)
2. Run all cells. Each sub-stage checkpoints, so partial progress resumes cleanly on re-run.
3. At the end, go to File → Save Version → Save & Run All. This materializes the output directory as a Kaggle dataset.
4. In your 07b notebook, use Add Data → Your Datasets to attach the output dataset produced here.

Once Part 1 finishes, run Part 2 (07b_training.ipynb).


## 0. Setup

In [ ]:
# ==== Dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip_install("pvlib", "properscoring", "pyarrow", "tqdm")

# ==== Environment detection ====
import os, time, json, shutil, gc, math
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

for d in [PERSIST_DIR, WORK_DIR,
          PERSIST_DIR / "checkpoints", PERSIST_DIR / "results",
          PERSIST_DIR / "latents",     PERSIST_DIR / "splits",
          PERSIST_DIR / "extended",    PERSIST_DIR / "figures"]:
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR        = WORK_DIR / "data"
CHECKPOINT_DIR  = PERSIST_DIR / "checkpoints"
RESULTS_DIR     = PERSIST_DIR / "results"
LATENT_DIR      = PERSIST_DIR / "latents"
SPLITS_DIR      = PERSIST_DIR / "splits"
EXTENDED_DIR    = PERSIST_DIR / "extended"
FIGURES_DIR     = PERSIST_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")

# ==== Clean-start toggle (set True on FIRST Kaggle run or when switching to v2 arch) ====
# v1 checkpoints are incompatible with v2 architecture; clearing them forces retrain.
CLEAN_START_V2 = False    # flip to True ONCE if switching from v1 -> v2
if CLEAN_START_V2:
    print("CLEAN_START_V2=True: removing old v1 checkpoints + results ...")
    for f in ["checkpoints/sde_best.pt", "checkpoints/sde_final.pt",
              "checkpoints/score_best.pt", "checkpoints/score_final.pt",
              "checkpoints/sde_a2_best.pt", "checkpoints/sde_a5_best.pt",
              "checkpoints/linear_decoder_a4.pt",
              "results/solar_sde_main_results.csv",
              "results/main_results_combined.csv",
              "results/ablation_results.csv",
              "results/solar_sde_calibrated.csv"]:
        p = PERSIST_DIR / f
        if p.exists():
            p.unlink()
            print(f"  removed {p.name}")
    print("Clean slate ready — Stage 0 will retrain with v2.")

# ==== GPU setup ====
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:  {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: CPU only. This notebook needs a GPU; please enable one.")
print(f"Using device: {DEVICE}")


In [ ]:
# ==== Soft fast-start: pull cached artifacts from GitHub if available ====
# Never raises — if anything is missing, the from-scratch stages below will
# produce it. This block exists purely as a fast path for users who don't
# want to spend 6+ hours retraining the Golden VAE.

import requests
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

def gh_pull_soft(rel_path, dest):
    if dest.exists() and dest.stat().st_size > 100:
        return True
    try:
        r = requests.get(f"{GITHUB_RAW}/{rel_path}", timeout=180)
        if r.status_code == 200 and len(r.content) > 100:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

print("Trying to pull cached upstream artifacts (best-effort, non-blocking) ...")
required = {
    CHECKPOINT_DIR / "vae_best.pt":   "colab_outputs/checkpoints/vae_best.pt",
    SPLITS_DIR    / "train.parquet":  "colab_outputs/splits/train.parquet",
    SPLITS_DIR    / "val.parquet":    "colab_outputs/splits/val.parquet",
    SPLITS_DIR    / "test.parquet":   "colab_outputs/splits/test.parquet",
    EXTENDED_DIR  / "train.parquet":  "colab_outputs/extended/train.parquet",
    EXTENDED_DIR  / "val.parquet":    "colab_outputs/extended/val.parquet",
    EXTENDED_DIR  / "test.parquet":   "colab_outputs/extended/test.parquet",
}
for split in ["train", "val", "test"]:
    for key in ["latents", "cti", "ghi", "covariates", "is_ramp", "kt", "ghi_clearsky",
                "physics_features"]:
        required[LATENT_DIR / f"{split}_{key}.npy"] = f"colab_outputs/latents/{split}_{key}.npy"
optional = {
    CHECKPOINT_DIR / "sde_best.pt":   "colab_outputs/checkpoints/sde_best.pt",
    CHECKPOINT_DIR / "score_best.pt": "colab_outputs/checkpoints/score_best.pt",
}
n_pulled, n_missing = 0, 0
for dest, rel in {**required, **optional}.items():
    if gh_pull_soft(rel, dest):
        if dest.exists():
            n_pulled += 1
    else:
        n_missing += 1

print(f"  pulled/already-present: {n_pulled}    missing: {n_missing}")
HAVE_VAE = (CHECKPOINT_DIR / "vae_best.pt").exists()
HAVE_SPLITS = (SPLITS_DIR / "train.parquet").exists()
HAVE_LATENTS = all((LATENT_DIR / f"{s}_latents.npy").exists() for s in ["train", "val", "test"])
HAVE_KT = all((LATENT_DIR / f"{s}_kt.npy").exists() for s in ["train", "val", "test"])
HAVE_PHYS = all((LATENT_DIR / f"{s}_physics_features.npy").exists() for s in ["train", "val", "test"])
HAVE_EXTENDED = (EXTENDED_DIR / "train.parquet").exists()

print(f"\nState of upstream artifacts:")
print(f"  VAE checkpoint:      {HAVE_VAE}")
print(f"  Splits parquets:     {HAVE_SPLITS}")
print(f"  Latents (z+cti):     {HAVE_LATENTS}")
print(f"  kt + ghi_clearsky:   {HAVE_KT}")
print(f"  Physics features:    {HAVE_PHYS}")
print(f"  Extended parquets:   {HAVE_EXTENDED}")

NEED_GOLDEN_RETRAIN = not (HAVE_VAE and HAVE_SPLITS and HAVE_LATENTS and HAVE_KT and HAVE_PHYS)
if NEED_GOLDEN_RETRAIN:
    print("\n[INFO] Some Golden artifacts missing — RETRAIN_GOLDEN stage will produce them.")
else:
    print("\n[INFO] All Golden artifacts present — RETRAIN_GOLDEN stage will skip.")


## 1. Shared model definitions

In [ ]:
# ==== Shared model definitions (matches Notebooks 1 + 2) ====

# --- CS-VAE (needed only for sanity; not retrained here) ---
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([nn.Conv2d(in_ch, ch, 4, 2, 1),
                           nn.GroupNorm(min(32, ch), ch),
                           nn.SiLU(inplace=True)])
            in_ch = ch
        self.conv = nn.Sequential(*layers); self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

# --- Neural SDE ---
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c): return self.net(torch.cat([z, t, c], dim=-1))

SIGMA_FLOOR_BASE = 0.01

class CTIDiffNet(nn.Module):
    """v2: diffusion floor + CTI scaling. sigma = floor(1+10*cti) + learned_softplus"""
    def __init__(self, z_dim=64, h=64, sigma_floor=SIGMA_FLOOR_BASE):
        super().__init__()
        self.sigma_floor = sigma_floor
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        base_floor = self.sigma_floor * (1.0 + 10.0 * cti)
        learned = self.out(self.state(z) * self.cti_gate(cti))
        return base_floor + learned

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim; self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c); sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        # v2: log-space diffusion matching (well-conditioned, prevents sigma collapse)
        resid = (zn - z - mu * dt).pow(2) / dt + 1e-8
        log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
        return {"loss": drift_l + self.lambda_sigma * log_diff_l,
                "drift": drift_l, "diffusion": log_diff_l}

# --- Score Decoder v3 (RESIDUAL prediction: delta_kt = kt(t+h) - kt(t)) ---
#
# v2 predicted absolute kt(t+h). For stable conditions where kt(t+h) ≈ kt(t),
# the model had to learn a near-identity mapping — neural nets are bad at this.
#
# v3 predicts delta_kt = kt(t+h) - kt(t). Targets are concentrated near 0
# (most timesteps have small change). At sampling time, we add the sampled
# delta to the current kt to get the prediction:
#
#   kt(t+h)_predicted = kt(t)_observed + delta_kt_sampled
#   GHI(t+h) = kt(t+h)_predicted * ghi_clearsky(t+h)
#
# This is the persistence-anchored parameterization. Default behavior is
# "no change" (delta=0 = persistence). Model learns to deviate from
# persistence only when context says so.

GHI_SCALE = 1200.0
KT_SCALE = 1.5
DELTA_KT_SCALE = 1.0    # delta_kt typically in [-1.0, 1.0], rarely outside

class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        # Inputs: (noisy_target, s, z, cti, c, kt_current)
        d_in = 1 + 1 + z_dim + 1 + c_dim + 1
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c, kt_cur):
        return self.net(torch.cat([g, s, z, cti, c, kt_cur], dim=-1))

class CondScoreDecoder(nn.Module):
    """v3: predicts delta_kt with persistence anchoring.

    Default mode (predict_mode='delta'):
      target = kt(t+h) - kt(t)
      sample: kt(t+h) = kt(t) + delta_sampled
    Other modes (legacy):
      'kt'  : predicts kt(t+h) directly (v2)
      'ghi' : predicts GHI(t+h) directly (v1)
    """
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02,
                 predict_mode='delta'):
        super().__init__()
        self.steps = steps
        self.predict_mode = predict_mode
        if predict_mode == 'delta':
            self.target_scale = DELTA_KT_SCALE
        elif predict_mode == 'kt':
            self.target_scale = KT_SCALE
        else:
            self.target_scale = GHI_SCALE
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps); alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))

    def _normalize(self, y):
        # For delta, scale [-DELTA_KT_SCALE, DELTA_KT_SCALE] -> [-1, 1]
        if self.predict_mode == 'delta':
            return y.clamp(-self.target_scale, self.target_scale) / self.target_scale
        else:
            return y / self.target_scale * 2.0 - 1.0
    def _denormalize(self, y):
        if self.predict_mode == 'delta':
            return y * self.target_scale
        else:
            return (y + 1.0) / 2.0 * self.target_scale

    def training_loss(self, kt_target, kt_current, z, cti, c):
        """Train on residual (or absolute, depending on mode)."""
        if self.predict_mode == 'delta':
            target_raw = kt_target - kt_current
        elif self.predict_mode == 'kt':
            target_raw = kt_target
        else:
            target_raw = kt_target  # caller passes ghi values in this mode
        t_norm = self._normalize(target_raw)
        B = t_norm.shape[0]
        si = torch.randint(0, self.steps, (B,), device=t_norm.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(t_norm)
        ts = self.sac[si].unsqueeze(-1) * t_norm + self.s1mac[si].unsqueeze(-1) * eps
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        pred_noise = self.score(ts, sn, z, cti, c, kt_cur_in)
        return {"loss": F.mse_loss(pred_noise, eps)}

    @torch.no_grad()
    def sample(self, z, cti, c, kt_current, n=1):
        """Returns samples in kt-space.
           predict_mode='delta': returns kt(t+h) = kt_current + delta_sampled (clamped to [0, KT_SCALE])
           predict_mode='kt'   : returns kt(t+h) directly
           predict_mode='ghi'  : returns GHI(t+h) directly (caller doesn't multiply by gcs)
        """
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        kt_cur_e = kt_cur_in.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            eps_pred = self.score(x, sn, z_e, cti_e, c_e, kt_cur_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x - b / (1 - ac).sqrt() * eps_pred)
            if i > 0: x = mean + b.sqrt() * torch.randn_like(x)
            else:     x = mean
        y_unscaled = self._denormalize(x)   # in target space (delta_kt or kt or ghi)
        if self.predict_mode == 'delta':
            kt_out = (kt_cur_e + y_unscaled).clamp(0.0, KT_SCALE)
        elif self.predict_mode == 'kt':
            kt_out = y_unscaled.clamp(0.0, KT_SCALE)
        else:
            kt_out = y_unscaled.clamp(0.0, GHI_SCALE)
        return kt_out.view(B, n)

# --- Metrics ---
def crps_empirical(y_true, y_samples):
    """y_true: (N,), y_samples: (N, M). Returns per-point CRPS (N,)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp_metric(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw_metric(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    if len(y_true) == 0: return {"crps": 0, "picp": 0, "pinaw": 0, "rmse": 0, "mae": 0, "ramp_crps": 0}
    y_med = np.median(y_samples, axis=1)
    y_range = float(y_true.max() - y_true.min())
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps":  float(crps.mean()),
        "picp":  picp_metric(y_true, y_samples, alpha),
        "pinaw": pinaw_metric(y_samples, y_range, alpha),
        "rmse":  float(np.sqrt(np.mean((y_true - y_med) ** 2))),
        "mae":   float(np.mean(np.abs(y_true - y_med))),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    else:
        out["ramp_crps"] = 0.0
    return out

# --- SDE solver (with stability clamping) ---
_train_Z_np = np.load(LATENT_DIR / "train_latents.npy")
Z_MEAN = torch.from_numpy(_train_Z_np.mean(0)).float().to(DEVICE)
Z_STD_RAW = torch.from_numpy(_train_Z_np.std(0)).float().to(DEVICE) + 1e-6
Z_STD = torch.maximum(Z_STD_RAW, torch.full_like(Z_STD_RAW, 0.05))
Z_CLAMP_STDS = 8.0
MU_CAP = 10.0
SIGMA_CAP = 5.0
del _train_Z_np

def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c).clamp(-MU_CAP, MU_CAP)
    sigma = diff_fn(z, cti).clamp(0.0, SIGMA_CAP)
    z_new = z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)
    return torch.clamp(z_new, Z_MEAN - Z_CLAMP_STDS * Z_STD, Z_MEAN + Z_CLAMP_STDS * Z_STD)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    """v4: with mixed-horizon training, the drift takes (z, normalized_horizon, c).
    At inference, we pass normalized_horizon = current_step / MAX_HORIZON as time input.
    This matches how the SDE was trained (drift(z, k/180, c) -> dz/k).
    The EM step uses physical dt=1.0; drift output is already in per-step units.
    """
    B, d = z0.shape
    mx = max(horizons); hset = set(horizons)
    MAX_HORIZON = 180.0
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t_norm = torch.full((B * N, 1), (step + 1) / MAX_HORIZON, device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t_norm, c_e, cti_e, dt)
        if (step + 1) in hset: out[step + 1] = z.view(B, N, d).clone()
    return out

print("Shared code loaded.")


## RETRAIN — Golden CO from raw data (skipped if all artifacts already present)

In [ ]:
# ==== Conditional guards on the Golden retrain blocks below ====
# Each Notebook 1 block (CLOUDCV_DOWNLOAD/EXTRACT/BMS/PREPROCESS/VAE/LATENT)
# has its own internal "skip if output exists" check. The wrapper here just
# documents the intent and lets you skip the whole stage with one toggle.

ENABLE_GOLDEN_RETRAIN = NEED_GOLDEN_RETRAIN   # auto-detected by fast-start
if not ENABLE_GOLDEN_RETRAIN:
    print("[SKIP] Golden retrain disabled (all artifacts present).")


In [ ]:
LATENT_DIM = 64
IMG_SIZE = 128
# ==== CS-VAE model definition ====
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([
                nn.Conv2d(in_ch, ch, 4, 2, 1),
                nn.GroupNorm(min(32, ch), ch),
                nn.SiLU(inplace=True),
            ])
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(256, 128, 64, 32)):
        super().__init__()
        self.init_ch = channels[0]
        self.fc = nn.Linear(latent_dim, channels[0] * 8 * 8)
        layers = []
        for i in range(len(channels) - 1):
            layers.extend([
                nn.ConvTranspose2d(channels[i], channels[i+1], 4, 2, 1),
                nn.GroupNorm(min(32, channels[i+1]), channels[i+1]),
                nn.SiLU(inplace=True),
            ])
        layers.extend([nn.ConvTranspose2d(channels[-1], 3, 4, 2, 1), nn.Sigmoid()])
        self.deconv = nn.Sequential(*layers)
    def forward(self, z):
        h = self.fc(z).view(-1, self.init_ch, 8, 8)
        return self.deconv(h)

class CloudStateVAE(nn.Module):
    def __init__(self, latent_dim=64, beta=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.beta = beta
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    def reparam(self, mu, lv):
        return mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
    def forward(self, x):
        mu, lv = self.encoder(x)
        z = self.reparam(mu, lv)
        return self.decoder(z), mu, lv
    def loss(self, x, recon, mu, lv):
        rec = F.mse_loss(recon, x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return {"loss": rec + self.beta * kl, "recon": rec, "kl": kl}
    @torch.no_grad()
    def encode_mu(self, x):
        mu, _ = self.encoder(x)
        return mu

n_params = sum(p.numel() for p in CloudStateVAE().parameters())
print(f"CS-VAE parameters: {n_params:,}")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not all((DATA_DIR / 'cloudcv' / f).exists() for f in ['2019_09_07.tar.gz']):
    # ==== Download CloudCV dataset ====
    import requests
    from tqdm import tqdm

    CLOUDCV_DIR = DATA_DIR / "cloudcv"
    CLOUDCV_DIR.mkdir(parents=True, exist_ok=True)

    CLOUDCV_FILES = {
        "README.md": "https://data.nlr.gov/system/files/248/1727758900-README.md",
        "2019_09_07.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_07.tar.gz",
        "2019_09_08.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_08.tar.gz",
        "2019_09_14.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_14.tar.gz",
        "2019_09_15.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_15.tar.gz",
        "2019_09_21.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_21.tar.gz",
        "2019_09_22.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_22.tar.gz",
        "2019_09_28.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_28.tar.gz",
        "2019_09_29.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_29.tar.gz",
    }

    def download_file(url, dest):
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"  Already have: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return True
        print(f"  Downloading {dest.name} ...")
        r = requests.get(url, stream=True, timeout=600, allow_redirects=True)
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk); pbar.update(len(chunk))
        return True

    print("=" * 70)
    print("Downloading CloudCV dataset (8 days of sky images + irradiance)")
    print("Expected total: ~2.6 GB")
    print("=" * 70)
    for name, url in CLOUDCV_FILES.items():
        download_file(url, CLOUDCV_DIR / name)
    print("CloudCV download complete.")


In [ ]:
if ENABLE_GOLDEN_RETRAIN:
    # ==== Extract CloudCV archives ====
    import tarfile

    print("Extracting tar.gz archives...")
    for tgz in sorted(CLOUDCV_DIR.glob("2019_*.tar.gz")):
        stem = tgz.stem.replace(".tar", "")   # 2019_09_07
        out = CLOUDCV_DIR / stem
        if out.exists() and any(out.rglob("*.jpg")):
            n = sum(1 for _ in (out / "images").glob("*.jpg"))
            print(f"  {stem}: already extracted ({n} images)")
            continue
        out.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tgz, "r:gz") as tf:
            tf.extractall(out)
        n_imgs = sum(1 for _ in (out / "images").glob("*.jpg"))
        print(f"  {stem}: extracted ({n_imgs} images)")

    # Free disk by removing tarballs after extraction
    for tgz in CLOUDCV_DIR.glob("*.tar.gz"):
        try:
            tgz.unlink()
        except Exception:
            pass
    print("Extraction complete. tar.gz archives removed to free disk.")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (DATA_DIR / 'bms' / 'bms_srrl_2019.csv').exists():
    # ==== Download BMS meteorological data (90-day period) ====
    BMS_DIR = DATA_DIR / "bms"
    BMS_DIR.mkdir(parents=True, exist_ok=True)
    BMS_PATH = BMS_DIR / "bms_srrl_2019.csv"

    BMS_URL = "https://midcdmz.nrel.gov/apps/data_api.pl?site=BMS&begin=20190905&end=20191203&inst=1&type=data"

    if BMS_PATH.exists() and BMS_PATH.stat().st_size > 10_000_000:
        print(f"BMS data already cached: {BMS_PATH.stat().st_size/1e6:.1f} MB")
    else:
        print(f"Downloading BMS 1-minute data from NREL MIDC API ...")
        r = requests.get(BMS_URL, timeout=600)
        r.raise_for_status()
        BMS_PATH.write_text(r.text)
        print(f"Saved: {BMS_PATH.stat().st_size/1e6:.1f} MB")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (SPLITS_DIR / 'train.parquet').exists():
    # ==== Preprocessing: parse CloudCV + BMS, align, compute features ====
    import numpy as np
    import pandas as pd
    from datetime import datetime
    import pvlib
    from pvlib.location import Location

    SRRL = Location(latitude=39.742, longitude=-105.18, tz="America/Denver",
                    altitude=1829, name="NREL SRRL")

    def parse_ts(s):
        s = s.strip()
        if s.startswith("UTC-7_"):
            s = s[6:]
        date_p, time_p = s.split("-")
        y, mo, d = date_p.split("_")
        tp = time_p.split("_")
        h, mi, sec = tp[0], tp[1], tp[2]
        us = tp[3] if len(tp) > 3 else "0"
        return datetime(int(y), int(mo), int(d), int(h), int(mi), int(sec), int(us))

    def load_cloudcv_day(day_dir):
        csv = day_dir / "pyranometer.csv"
        imgs = day_dir / "images"
        if not csv.exists():
            return pd.DataFrame()
        rows = []
        for line in open(csv):
            line = line.strip()
            if not line or "," not in line:
                continue
            parts = line.split(",")
            if len(parts) < 2:
                continue
            try:
                ts = parse_ts(parts[0])
            except Exception:
                continue
            mv = float(parts[1].strip())
            img_name = parts[0].strip() + ".jpg"
            img_path = imgs / img_name
            rows.append({
                "timestamp": ts, "millivolts": mv,
                "image_path": str(img_path),
                "image_exists": img_path.exists(),
            })
        df = pd.DataFrame(rows)
        if len(df) > 0:
            df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df

    print("Loading CloudCV days ...")
    day_dirs = sorted([d for d in CLOUDCV_DIR.iterdir() if d.is_dir() and d.name.startswith("2019")])
    all_dfs = []
    for d in day_dirs:
        df = load_cloudcv_day(d)
        if len(df) > 0:
            all_dfs.append(df)
            print(f"  {d.name}: {len(df)} rows ({df['image_exists'].sum()} images)")

    cloudcv = pd.concat(all_dfs, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
    print(f"Total CloudCV rows: {len(cloudcv)}")

    print("\nLoading BMS ...")
    bms_raw = pd.read_csv(BMS_PATH)
    print(f"  BMS rows: {len(bms_raw)}")
    print(f"  BMS columns: {len(bms_raw.columns)}")

    # Build BMS timestamps from Year, DOY, MST
    ts_list = []
    for _, r in bms_raw.iterrows():
        try:
            y, doy, mst = int(r["Year"]), int(r["DOY"]), int(r["MST"])
            dt = datetime.strptime(f"{y}-{doy}", "%Y-%j").replace(hour=mst // 60, minute=mst % 60)
            ts_list.append(dt)
        except Exception:
            ts_list.append(pd.NaT)
    bms_raw["timestamp"] = pd.to_datetime(ts_list)

    bms = pd.DataFrame({
        "timestamp": bms_raw["timestamp"],
        "ghi_bms": bms_raw.get("Global LI-200 [W/m^2]"),
        "dni_bms": bms_raw.get("Direct NIP [W/m^2]"),
        "dhi_bms": bms_raw.get("Diffuse CM22-1 (vent/cor) [W/m^2]"),
        "temperature": bms_raw.get("Deck Dry Bulb Temp [deg C]"),
        "humidity": bms_raw.get("Deck RH [%]"),
        "wind_speed": bms_raw.get("Avg Wind Speed @ 19ft [m/s]"),
        "pressure": bms_raw.get("Station Pressure [mBar]"),
        "cloud_cover_total": bms_raw.get("Total Cloud Cover [%]"),
    })
    for c in bms.columns:
        if c != "timestamp":
            bms[c] = pd.to_numeric(bms[c], errors="coerce").replace([-7999, -6999, -9999], np.nan)

    print(f"  BMS GHI range: [{bms['ghi_bms'].min():.1f}, {bms['ghi_bms'].max():.1f}] W/m²")

    # Interpolate BMS GHI to 10s
    print("\nInterpolating BMS GHI to 10-second resolution ...")
    bms_ghi = bms[["timestamp", "ghi_bms"]].dropna().copy()
    bms_ghi = bms_ghi.set_index("timestamp").sort_index()
    bms_10s = bms_ghi.resample("10s").interpolate(method="linear")

    cloudcv["ts_round"] = cloudcv["timestamp"].dt.round("10s")
    ghi_vals = []
    for ts in cloudcv["ts_round"]:
        if ts in bms_10s.index:
            ghi_vals.append(float(bms_10s.loc[ts, "ghi_bms"]))
        else:
            i = bms_10s.index.get_indexer([ts], method="nearest")[0]
            ghi_vals.append(float(bms_10s.iloc[i]["ghi_bms"]) if 0 <= i < len(bms_10s) else np.nan)
    cloudcv["ghi"] = np.clip(ghi_vals, 0, None)

    # Merge meteo covariates
    cloudcv["ts_minute"] = cloudcv["timestamp"].dt.floor("min")
    bms["ts_minute"] = bms["timestamp"].dt.floor("min")
    merged = cloudcv.merge(bms.drop(columns=["timestamp"]), on="ts_minute", how="left")
    for c in ["temperature", "humidity", "wind_speed", "pressure", "cloud_cover_total"]:
        if c in merged.columns:
            merged[c] = merged[c].ffill().fillna(0)

    # Solar geometry + clear sky
    print("\nComputing solar geometry + clear-sky ...")
    tz_ts = pd.DatetimeIndex(merged["timestamp"]).tz_localize("America/Denver")
    solpos = SRRL.get_solarposition(tz_ts)
    merged["solar_zenith"] = solpos["apparent_zenith"].values
    cs = SRRL.get_clearsky(tz_ts, model="ineichen")
    merged["ghi_clearsky"] = cs["ghi"].values
    with np.errstate(divide="ignore", invalid="ignore"):
        kt = merged["ghi"].values / merged["ghi_clearsky"].values
        kt = np.where(merged["ghi_clearsky"].values < 10, 0.0, kt)
    merged["clear_sky_index"] = np.clip(kt, 0, 1.5)

    # Quality filter: daytime, image exists, valid GHI
    before = len(merged)
    merged = merged[(merged["solar_zenith"] <= 85.0) & (merged["ghi"] >= 0)
                    & (merged["ghi"].notna()) & (merged["image_exists"])].reset_index(drop=True)
    print(f"Quality filter: {before} -> {len(merged)} rows")

    # Ramp detection (|ΔGHI| > 50 W/m² in 60s)
    dg = merged["ghi"].diff(6).abs() / 1.0
    merged["is_ramp"] = (dg > 50.0).fillna(False)
    print(f"Ramp events: {int(merged['is_ramp'].sum())} ({merged['is_ramp'].mean()*100:.1f}%)")

    # Chronological split (5 train / 1 val / 2 test by date)
    dates = sorted(merged["timestamp"].dt.date.unique())
    print(f"\nUnique dates: {len(dates)}")
    n_tr = max(1, int(len(dates) * 0.625))
    n_val = max(1, int(len(dates) * 0.125))
    train_dates = set(dates[:n_tr])
    val_dates = set(dates[n_tr:n_tr + n_val])
    test_dates = set(dates[n_tr + n_val:])

    train_df = merged[merged["timestamp"].dt.date.isin(train_dates)].reset_index(drop=True)
    val_df   = merged[merged["timestamp"].dt.date.isin(val_dates)].reset_index(drop=True)
    test_df  = merged[merged["timestamp"].dt.date.isin(test_dates)].reset_index(drop=True)

    SPLITS_DIR = PERSIST_DIR / "splits"
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    train_df.to_parquet(SPLITS_DIR / "train.parquet")
    val_df.to_parquet(SPLITS_DIR / "val.parquet")
    test_df.to_parquet(SPLITS_DIR / "test.parquet")

    print(f"\nSplit sizes:")
    print(f"  train: {len(train_df):>6} rows ({len(train_dates)} days)")
    print(f"  val:   {len(val_df):>6} rows ({len(val_dates)} days)")
    print(f"  test:  {len(test_df):>6} rows ({len(test_dates)} days)")
    print(f"  GHI range overall: [{merged['ghi'].min():.1f}, {merged['ghi'].max():.1f}] W/m²")


In [ ]:
if ENABLE_GOLDEN_RETRAIN:
    # ==== Image Dataset (loads JPEGs on the fly) ====
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    import numpy as np
    import pandas as pd

    def load_img(path, size=128):
        img = Image.open(path).convert("RGB")
        w, h = img.size
        side = min(w, h)
        l, t = (w - side) // 2, (h - side) // 2
        img = img.crop((l, t, l + side, t + side)).resize((size, size), Image.BILINEAR)
        return np.array(img, dtype=np.float32) / 255.0

    class SkyImageDataset(Dataset):
        def __init__(self, parquet_path, size=128):
            self.df = pd.read_parquet(parquet_path)
            if "image_exists" in self.df.columns:
                self.df = self.df[self.df["image_exists"]].reset_index(drop=True)
            self.size = size
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            p = self.df.iloc[i]["image_path"]
            arr = load_img(p, self.size)
            return torch.from_numpy(arr).permute(2, 0, 1)

    for sp in ["train", "val", "test"]:
        ds = SkyImageDataset(SPLITS_DIR / f"{sp}.parquet")
        print(f"  {sp}: {len(ds)} images")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (CHECKPOINT_DIR / 'vae_best.pt').exists():
    # ==== Train CS-VAE ====
    from tqdm import tqdm
    import time

    IMG_SIZE = 128
    LATENT_DIM = 64
    BATCH = 32
    EPOCHS = 20           # trimmed from 100 — sufficient for 128x128 with this dataset size
    LR = 1e-4
    SEED = 42

    torch.manual_seed(SEED); np.random.seed(SEED)

    train_ds = SkyImageDataset(SPLITS_DIR / "train.parquet", size=IMG_SIZE)
    val_ds   = SkyImageDataset(SPLITS_DIR / "val.parquet",   size=IMG_SIZE)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

    model = CloudStateVAE(latent_dim=LATENT_DIM, beta=0.1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    print(f"Training VAE: {EPOCHS} epochs, batch={BATCH}, img={IMG_SIZE}x{IMG_SIZE}, latent_dim={LATENT_DIM}")
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    print("=" * 70)

    best_val = float("inf")
    history = []
    t_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tl, tr, tk = 0, 0, 0
        for img in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            img = img.to(DEVICE, non_blocking=True)
            rec, mu, lv = model(img)
            losses = model.loss(img, rec, mu, lv)
            opt.zero_grad(); losses["loss"].backward(); opt.step()
            tl += losses["loss"].item(); tr += losses["recon"].item(); tk += losses["kl"].item()
        tl /= len(train_loader); tr /= len(train_loader); tk /= len(train_loader)

        model.eval()
        vl = 0
        with torch.no_grad():
            for img in val_loader:
                img = img.to(DEVICE, non_blocking=True)
                rec, mu, lv = model(img)
                vl += model.loss(img, rec, mu, lv)["loss"].item()
        vl /= max(len(val_loader), 1)

        elapsed = (time.time() - t_start) / 60
        print(f"Epoch {epoch:3d}/{EPOCHS} | train={tl:.4f} (rec={tr:.4f}, kl={tk:.4f}) | val={vl:.4f} | {elapsed:.1f} min")
        history.append({"epoch": epoch, "train_loss": tl, "train_recon": tr, "train_kl": tk, "val_loss": vl})

        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), CHECKPOINT_DIR / "vae_best.pt")
            print(f"  [best] saved checkpoint (val={vl:.4f})")

    torch.save(model.state_dict(), CHECKPOINT_DIR / "vae_final.pt")
    pd.DataFrame(history).to_csv(RESULTS_DIR / "vae_training_history.csv", index=False)
    print("=" * 70)
    print(f"VAE training complete. Best val loss: {best_val:.4f}. Total time: {(time.time()-t_start)/60:.1f} min")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (LATENT_DIR / 'test_latents.npy').exists():
    # ==== Extract latents + compute CTI ====
    from torch.utils.data import DataLoader

    def cti_from_latents(Z, window=10):
        """Compute CTI as L2 norm of variance of latent velocity over a sliding window."""
        T = Z.shape[0]
        cti = np.zeros(T, dtype=np.float32)
        for t in range(window, T):
            win = Z[t - window:t]
            v = np.diff(win, axis=0)
            var = v.var(axis=0)
            cti[t] = np.linalg.norm(var, ord=2)
        return cti

    # Load best VAE
    model = CloudStateVAE(latent_dim=LATENT_DIM, beta=0.1).to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / "vae_best.pt", map_location=DEVICE))
    model.eval()

    print("Extracting latents for train / val / test ...")
    for split in ["train", "val", "test"]:
        ds = SkyImageDataset(SPLITS_DIR / f"{split}.parquet", size=IMG_SIZE)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
        all_mu = []
        with torch.no_grad():
            for img in tqdm(loader, desc=f"Encoding {split}"):
                img = img.to(DEVICE, non_blocking=True)
                all_mu.append(model.encode_mu(img).cpu().numpy())
        Z = np.concatenate(all_mu, axis=0).astype(np.float32)
        cti = cti_from_latents(Z, window=10)

        df = ds.df
        ghi = df["ghi"].values.astype(np.float32)
        cov_cols = [c for c in ["solar_zenith", "clear_sky_index", "temperature",
                                "humidity", "wind_speed"] if c in df.columns]
        cov = df[cov_cols].fillna(0).values.astype(np.float32) if cov_cols else np.zeros((len(df), 0), np.float32)

        np.save(LATENT_DIR / f"{split}_latents.npy", Z)
        np.save(LATENT_DIR / f"{split}_cti.npy",     cti)
        np.save(LATENT_DIR / f"{split}_ghi.npy",     ghi)
        np.save(LATENT_DIR / f"{split}_covariates.npy", cov)
        np.save(LATENT_DIR / f"{split}_is_ramp.npy", df["is_ramp"].values.astype(bool))

        print(f"  {split}: Z={Z.shape}, CTI range=[{cti.min():.4f}, {cti.max():.4f}], "
              f"GHI range=[{ghi.min():.1f}, {ghi.max():.1f}], covariates={cov.shape}")
    print("Latent extraction complete.")


In [ ]:
# ==== Save kt + ghi_clearsky + physics features for each split ====
# Required by LOAD_DATA: kt, ghi_clearsky come from the parquet columns.
# Physics features (15-dim) are computed inline from the split parquets.

import numpy as np, pandas as pd

ALL_KT_DONE = all((LATENT_DIR / f"{s}_kt.npy").exists() and
                  (LATENT_DIR / f"{s}_ghi_clearsky.npy").exists()
                  for s in ["train", "val", "test"])
if ALL_KT_DONE:
    print("[SKIP] kt + ghi_clearsky already saved.")
else:
    print("Saving kt + ghi_clearsky from split parquets ...")
    for split in ["train", "val", "test"]:
        df = pd.read_parquet(SPLITS_DIR / f"{split}.parquet")
        if "image_exists" in df.columns:
            df = df[df["image_exists"]].reset_index(drop=True)
        kt = df["clear_sky_index"].values.astype(np.float32)
        gcs = df["ghi_clearsky"].values.astype(np.float32)
        np.save(LATENT_DIR / f"{split}_kt.npy", kt)
        np.save(LATENT_DIR / f"{split}_ghi_clearsky.npy", gcs)
        print(f"  {split}: kt range [{kt.min():.3f}, {kt.max():.3f}], gcs range [{gcs.min():.1f}, {gcs.max():.1f}]")

ALL_PHYS_DONE = all((LATENT_DIR / f"{s}_physics_features.npy").exists()
                    for s in ["train", "val", "test"])
if ALL_PHYS_DONE:
    print("[SKIP] Physics features already computed.")
else:
    print("\nComputing physics features (15-dim per row) ...")
    def compute_physics_15(df):
        df = df.reset_index(drop=True).copy()
        n = len(df)
        zenith = df["solar_zenith"].values.astype(np.float32)
        zenith_clip = np.clip(zenith, 0, 89.9)
        zenith_rad = np.deg2rad(zenith_clip)
        air_mass = 1.0 / (np.cos(zenith_rad) + 0.50572 * (96.07995 - zenith_clip) ** -1.6364)
        air_mass = np.clip(air_mass, 1.0, 40.0).astype(np.float32)
        zenith_rate = np.zeros(n, dtype=np.float32)
        zenith_rate[1:] = (zenith[1:] - zenith[:-1]) / 10.0
        az = df.get("solar_azimuth", pd.Series(np.zeros(n))).values.astype(np.float32)
        az_rad = np.deg2rad(az)
        ts = pd.to_datetime(df["timestamp"])
        hour_frac = (ts.dt.hour + ts.dt.minute / 60.0 + ts.dt.second / 3600.0).values
        doy = ts.dt.dayofyear.values
        kt = df["clear_sky_index"].values.astype(np.float32)
        def trend(s, lag):
            o = np.zeros_like(s); o[lag:] = (s[lag:] - s[:-lag]) / lag; return o
        def rstd(s, w):
            o = np.zeros_like(s)
            for i in range(w, len(s)):
                o[i] = np.std(s[i-w:i])
            return o
        ghi = df["ghi"].values.astype(np.float32)
        pyr = df["millivolts"].values.astype(np.float32) if "millivolts" in df.columns else np.zeros(n, np.float32)
        return np.stack([
            air_mass, zenith_rate,
            np.sin(az_rad).astype(np.float32), np.cos(az_rad).astype(np.float32),
            np.sin(2*np.pi*hour_frac/24).astype(np.float32),
            np.cos(2*np.pi*hour_frac/24).astype(np.float32),
            np.sin(2*np.pi*doy/365.25).astype(np.float32),
            np.cos(2*np.pi*doy/365.25).astype(np.float32),
            np.clip((hour_frac - 6.0) / 12.0, 0, 1).astype(np.float32),
            trend(kt, 6), trend(kt, 30), trend(kt, 60),
            rstd(kt, 30).astype(np.float32),
            (rstd(ghi, 6) / 1200.0).astype(np.float32),
            pyr,
        ], axis=1)
    for split in ["train", "val", "test"]:
        df = pd.read_parquet(SPLITS_DIR / f"{split}.parquet")
        if "image_exists" in df.columns:
            df = df[df["image_exists"]].reset_index(drop=True)
        feats = compute_physics_15(df)
        np.save(LATENT_DIR / f"{split}_physics_features.npy", feats)
        print(f"  {split}: physics features shape={feats.shape}")


In [ ]:
# ==== Build extended (90-day BMS-only) splits for LSTM baselines ====
# These provide ~12x more training data for the LSTM/MC-Dropout/Deep-Ensemble/
# TimeGrad baselines. Without them, baselines train on only ~5 days of 10s data.

if HAVE_EXTENDED:
    print("[SKIP] Extended splits already present.")
else:
    print("Building extended (90-day BMS-only) splits ...")
    import pandas as pd, numpy as np

    BMS_PATH = DATA_DIR / "bms" / "bms_srrl_2019.csv"
    if not BMS_PATH.exists():
        print("[WARN] BMS data not present — extended splits cannot be built.")
        print("       Re-run BMS_DOWNLOAD cell first, or LSTM baselines will be limited.")
    else:
        try:
            import pvlib
            from pvlib.location import Location
        except ImportError:
            pip_install("pvlib")
            import pvlib
            from pvlib.location import Location
        SRRL = Location(latitude=39.742, longitude=-105.18, tz="America/Denver",
                        altitude=1829, name="NREL SRRL")

        bms_raw = pd.read_csv(BMS_PATH)
        from datetime import datetime
        ts_list = []
        for _, r in bms_raw.iterrows():
            try:
                y, doy, mst = int(r["Year"]), int(r["DOY"]), int(r["MST"])
                hh, mm = divmod(mst, 100)
                dt = datetime.strptime(f"{y}-{doy}", "%Y-%j").replace(hour=hh, minute=mm)
                ts_list.append(dt)
            except Exception:
                ts_list.append(pd.NaT)
        bms_raw["timestamp"] = pd.to_datetime(ts_list)
        bms_raw = bms_raw.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

        cols_map = {
            "ghi": "Global LI-200 [W/m^2]",
            "dni": "Direct NIP [W/m^2]",
            "dhi": "Diffuse CM22-1 (vent/cor) [W/m^2]",
            "temperature": "Deck Dry Bulb Temp [deg C]",
            "humidity":    "Deck RH [%]",
            "wind_speed":  "Avg Wind Speed @ 19ft [m/s]",
            "millivolts":  "Global LI-200 [W/m^2]",
        }
        out = pd.DataFrame({"timestamp": bms_raw["timestamp"]})
        for k, src in cols_map.items():
            out[k] = pd.to_numeric(bms_raw.get(src), errors="coerce").replace(
                [-7999, -6999, -9999], np.nan)
        out["ghi"] = out["ghi"].clip(lower=0)
        out = out.dropna(subset=["ghi"]).reset_index(drop=True)

        # Solar geometry + clear sky
        tz_ts = pd.DatetimeIndex(out["timestamp"]).tz_localize("America/Denver")
        sp = SRRL.get_solarposition(tz_ts)
        out["solar_zenith"] = sp["apparent_zenith"].values
        out["solar_azimuth"] = sp["azimuth"].values
        cs = SRRL.get_clearsky(tz_ts, model="ineichen")
        out["ghi_clearsky"] = cs["ghi"].values
        out["clear_sky_index"] = np.clip(out["ghi"] / out["ghi_clearsky"].replace(0, np.nan), 0, 1.5).fillna(0)
        out = out[out["solar_zenith"] <= 85.0].reset_index(drop=True)
        out["is_ramp"] = (out["ghi"].diff(1).abs() > 50.0).fillna(False)

        # Chronological 60/15/15 split by date
        dates = sorted(out["timestamp"].dt.date.unique())
        n_tr = int(len(dates) * 0.7); n_val = int(len(dates) * 0.15)
        tr_set = set(dates[:n_tr]); va_set = set(dates[n_tr:n_tr+n_val])
        ext_tr = out[out["timestamp"].dt.date.isin(tr_set)].reset_index(drop=True)
        ext_va = out[out["timestamp"].dt.date.isin(va_set)].reset_index(drop=True)
        ext_te = out[~out["timestamp"].dt.date.isin(tr_set | va_set)].reset_index(drop=True)
        ext_tr.to_parquet(EXTENDED_DIR / "train.parquet")
        ext_va.to_parquet(EXTENDED_DIR / "val.parquet")
        ext_te.to_parquet(EXTENDED_DIR / "test.parquet")
        print(f"  extended: train={len(ext_tr):,}  val={len(ext_va):,}  test={len(ext_te):,}")


## 2. Load data tensors

In [ ]:
# ==== Load all data tensors (tolerant: degrades gracefully if extended missing) ====
def load_split(s):
    orig_cov = np.load(LATENT_DIR / f"{s}_covariates.npy")
    phys = np.load(LATENT_DIR / f"{s}_physics_features.npy")
    img_feat_path = LATENT_DIR / f"{s}_image_features.npy"
    if img_feat_path.exists():
        img_feats = np.load(img_feat_path)
        cov = np.concatenate([orig_cov, phys, img_feats], axis=1).astype(np.float32)
    else:
        cov = np.concatenate([orig_cov, phys], axis=1).astype(np.float32)
    return {
        "Z":    np.load(LATENT_DIR / f"{s}_latents.npy"),
        "cti":  np.load(LATENT_DIR / f"{s}_cti.npy"),
        "ghi":  np.load(LATENT_DIR / f"{s}_ghi.npy"),
        "cov":  cov,
        "ramp": np.load(LATENT_DIR / f"{s}_is_ramp.npy"),
        "kt":   np.load(LATENT_DIR / f"{s}_kt.npy"),
        "gcs":  np.load(LATENT_DIR / f"{s}_ghi_clearsky.npy"),
    }
data = {s: load_split(s) for s in ["train", "val", "test"]}
print(f"\n  Covariate dim: {data['train']['cov'].shape[1]}  "
      f"(5 original + 15 physics + "
      f"{data['train']['cov'].shape[1] - 20} image features)")
for s, d in data.items():
    print(f"  {s}: Z={d['Z'].shape}, GHI=[{d['ghi'].min():.0f},{d['ghi'].max():.0f}], ramps={int(d['ramp'].sum())}")

train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
val_df   = pd.read_parquet(SPLITS_DIR / "val.parquet")
test_df  = pd.read_parquet(SPLITS_DIR / "test.parquet")
print(f"\n8-day image splits: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")

# Extended (90-day BMS) splits — used by LSTM/MC-Dropout/TimeGrad/Deep-Ensemble baselines.
# If missing, we fall back to using the regular train_df/val_df for those baselines.
HAVE_EXT = (EXTENDED_DIR / "train.parquet").exists() and (EXTENDED_DIR / "val.parquet").exists()
if HAVE_EXT:
    ext_train = pd.read_parquet(EXTENDED_DIR / "train.parquet")
    ext_val   = pd.read_parquet(EXTENDED_DIR / "val.parquet")
    print(f"90-day extended:    train={len(ext_train):,} val={len(ext_val):,}")
else:
    print("[WARN] Extended (90-day BMS) parquets missing — LSTM baselines will train on the")
    print("       8-day image splits instead, with reduced sample count.")
    # Fallback: replicate the structure expected by BASELINES_CODE
    ext_train = train_df.copy()
    ext_val   = val_df.copy()

Z_DIM = data["train"]["Z"].shape[1]
C_DIM = max(1, data["train"]["cov"].shape[1])
print(f"\nZ_DIM={Z_DIM}, C_DIM={C_DIM}")

HORIZONS = [6, 30, 60, 120, 180]
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
N_EVAL = min(1000, len(data["test"]["Z"]) - max(HORIZONS) - 1)
SEQ_LEN = 30
print(f"Horizons: {list(HORIZON_MIN.values())} min, MC samples: {N_SAMPLES}, N_EVAL: {N_EVAL}")


## STAGE A — Stanford SKIPP'D as full second site

In [ ]:
# ==== STAGE A: Stanford SKIPP'D as full second site ====
# Downloads the SKIPP'D HDF5, trains a separate VAE on its 64x64 images
# (upsampled to 128x128), trains a separate SDE+Score Decoder on Stanford's
# PV power as the forecast target, and runs forecast evaluation.
#
# Stanford SKIPP'D: 497 trainval days + ~100 test days at 1-min resolution.
# Target = PV power (kW from one panel). NOT GHI.
# This gives us a SECOND independent experimental result for the paper.

ENABLE_STANFORD = True   # set False to skip Stanford pipeline entirely

SF_DIR = WORK_DIR / "stanford_skippd"
SF_DIR.mkdir(parents=True, exist_ok=True)
SF_HDF5 = SF_DIR / "2017_2019_images_pv_processed.hdf5"
SF_TIMES_TV = SF_DIR / "times_trainval.npy"
SF_TIMES_TE = SF_DIR / "times_test.npy"

SF_VAE_CKPT = CHECKPOINT_DIR / "stanford_vae_best.pt"
SF_SDE_CKPT = CHECKPOINT_DIR / "stanford_sde_best.pt"
SF_SCORE_CKPT = CHECKPOINT_DIR / "stanford_score_best.pt"
SF_LATENTS_DIR = LATENT_DIR / "stanford"
SF_LATENTS_DIR.mkdir(parents=True, exist_ok=True)
SF_RESULTS_DIR = RESULTS_DIR / "stanford"
SF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STAGE_A_DONE = (SF_RESULTS_DIR / "solarsde_results.csv").exists()

if not ENABLE_STANFORD:
    print("[SKIP] Stage A disabled (ENABLE_STANFORD=False).")
elif STAGE_A_DONE:
    print("[SKIP] Stage A: Stanford pipeline already complete (results CSV exists).")
else:
    print("=" * 70)
    print("STAGE A: Stanford SKIPP'D full pipeline")
    print("=" * 70)
    pip_install("h5py")
    import h5py

    # ---- A.1 Download SKIPP'D HDF5 (~3-4 GB) ----
    if not SF_HDF5.exists() or SF_HDF5.stat().st_size < 1_000_000_000:
        print("[A.1] Downloading SKIPP'D HDF5 (~3-4 GB) ...")
        import requests
        url = "https://stacks.stanford.edu/file/dj417rh1007/2017_2019_images_pv_processed.hdf5"
        with requests.get(url, stream=True, timeout=3600) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(SF_HDF5, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="HDF5") as pb:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk); pb.update(len(chunk))
        # Times metadata
        for nm in ("times_trainval.npy", "times_test.npy"):
            r = requests.get(f"https://stacks.stanford.edu/file/dj417rh1007/{nm}", timeout=600)
            (SF_DIR / nm).write_bytes(r.content)
    else:
        print("[A.1] SKIPP'D HDF5 already present.")

    # ---- A.2 Load splits + persist as npy ----
    print("[A.2] Loading SKIPP'D HDF5 ...")
    with h5py.File(SF_HDF5, "r") as f:
        for split_key, prefix in [("trainval", "stanford_train"), ("test", "stanford_test")]:
            if split_key not in f:
                continue
            grp = f[split_key]
            img_key = "images_log" if "images_log" in grp else list(grp.keys())[0]
            pv_key = "pv_log" if "pv_log" in grp else next(k for k in grp.keys() if "pv" in k.lower())
            np.save(SF_DIR / f"{prefix}_images.npy", grp[img_key][:])
            np.save(SF_DIR / f"{prefix}_pv.npy", grp[pv_key][:])
    print("  saved per-split npy files")

    # Build train/val/test splits with timestamps
    sf_train_imgs = np.load(SF_DIR / "stanford_train_images.npy")
    sf_train_pv = np.load(SF_DIR / "stanford_train_pv.npy")
    sf_train_times = np.load(SF_DIR / "times_trainval.npy", allow_pickle=True)
    sf_test_imgs = np.load(SF_DIR / "stanford_test_images.npy")
    sf_test_pv = np.load(SF_DIR / "stanford_test_pv.npy")
    sf_test_times = np.load(SF_DIR / "times_test.npy", allow_pickle=True)

    # 80/20 split of trainval into train/val (chronological by date)
    import pandas as pd
    ts_tv = pd.to_datetime(sf_train_times)
    days_tv = sorted(set(ts_tv.normalize()))
    n_train_days = int(len(days_tv) * 0.8)
    train_day_set = set(days_tv[:n_train_days])
    train_mask = np.array([t.normalize() in train_day_set for t in ts_tv])
    val_mask = ~train_mask
    print(f"  Stanford: train={train_mask.sum()} val={val_mask.sum()} test={len(sf_test_pv)}")

    SF_PV_SCALE = float(np.percentile(sf_train_pv, 99))   # use 99th percentile for normalization
    print(f"  PV scale (99th pct): {SF_PV_SCALE:.2f} kW")
    np.save(SF_LATENTS_DIR / "pv_scale.npy", np.array([SF_PV_SCALE]))

    # ---- VAE architecture (inline, matches Notebook 1's STAGE_MINUS2 VAE) ----
    class _SfEnc(nn.Module):
        def __init__(self, latent=64, ch=(32, 64, 128, 256)):
            super().__init__(); L, ic = [], 3
            for c in ch:
                L += [nn.Conv2d(ic, c, 4, 2, 1), nn.GroupNorm(min(32, c), c), nn.SiLU(inplace=True)]
                ic = c
            self.conv = nn.Sequential(*L); self.pool = nn.AdaptiveAvgPool2d(1)
            self.fc_mu = nn.Linear(ch[-1], latent); self.fc_lv = nn.Linear(ch[-1], latent)
        def forward(self, x):
            h = self.pool(self.conv(x)).flatten(1); return self.fc_mu(h), self.fc_lv(h)
    class _SfDec(nn.Module):
        def __init__(self, latent=64, ch=(256, 128, 64, 32)):
            super().__init__(); self.init_ch = ch[0]
            self.fc = nn.Linear(latent, ch[0] * 8 * 8); L = []
            for i in range(len(ch) - 1):
                L += [nn.ConvTranspose2d(ch[i], ch[i+1], 4, 2, 1),
                      nn.GroupNorm(min(32, ch[i+1]), ch[i+1]), nn.SiLU(inplace=True)]
            L += [nn.ConvTranspose2d(ch[-1], 3, 4, 2, 1), nn.Sigmoid()]
            self.deconv = nn.Sequential(*L)
        def forward(self, z): return self.deconv(self.fc(z).view(-1, self.init_ch, 8, 8))
    class _SfVAE(nn.Module):
        def __init__(self, latent=64, beta=0.1):
            super().__init__(); self.beta = beta
            self.encoder = _SfEnc(latent); self.decoder = _SfDec(latent)
        def forward(self, x):
            mu, lv = self.encoder(x); z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
            return self.decoder(z), mu, lv
        def loss(self, x, rec, mu, lv):
            r = F.mse_loss(rec, x); k = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
            return r + self.beta * k

    # ---- A.3 Train Stanford VAE (independent) ----
    if not SF_VAE_CKPT.exists():
        print("[A.3] Training Stanford VAE (30 epochs, 128x128 upsampled) ...")
        class SfVAEDS(Dataset):
            def __init__(self, imgs, target=128):
                self.imgs = imgs; self.target = target
            def __len__(self): return len(self.imgs)
            def __getitem__(self, i):
                img = torch.from_numpy(self.imgs[i]).float() / 255.0
                if img.dim() == 3: img = img.permute(2, 0, 1)
                img = img.unsqueeze(0)
                img = F.interpolate(img, size=self.target, mode="bilinear", align_corners=False)
                return img.squeeze(0)
        ds = SfVAEDS(sf_train_imgs[train_mask])
        dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        torch.manual_seed(42)
        vae = _SfVAE(latent=64, beta=0.1).to(DEVICE)
        opt = torch.optim.Adam(vae.parameters(), lr=1e-4)
        best = float("inf")
        for ep in range(1, 31):
            vae.train(); tl = 0; n = 0
            for img in dl:
                img = img.to(DEVICE, non_blocking=True)
                recon, mu, lv = vae(img)
                loss = vae.loss(img, recon, mu, lv)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0); opt.step()
                tl += loss.item(); n += 1
            tl /= n
            print(f"  ep {ep}/30: loss={tl:.4f}")
            if tl < best:
                best = tl; torch.save(vae.state_dict(), SF_VAE_CKPT)
        del vae, ds, dl; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    else:
        print("[A.3] Stanford VAE checkpoint exists, skipping.")

    # ---- A.4 Encode Stanford latents + CTI per split ----
    if not (SF_LATENTS_DIR / "test_latents.npy").exists():
        print("[A.4] Encoding Stanford latents + CTI ...")
        vae = _SfVAE(latent=64).to(DEVICE)
        vae.load_state_dict(torch.load(SF_VAE_CKPT, map_location=DEVICE, weights_only=False))
        vae.eval()

        @torch.no_grad()
        def encode_imgs(imgs, batch=128):
            out = []
            for i in tqdm(range(0, len(imgs), batch), desc="enc"):
                b = imgs[i:i+batch]
                x = torch.from_numpy(b).float() / 255.0
                if x.dim() == 4: x = x.permute(0, 3, 1, 2)
                x = F.interpolate(x, size=128, mode="bilinear", align_corners=False).to(DEVICE)
                mu, _ = vae.encoder(x)
                out.append(mu.cpu().numpy())
            return np.concatenate(out, axis=0)

        def cti_window(z, w=10):
            n = len(z); cti = np.zeros(n, dtype=np.float32)
            for i in range(w, n):
                v = np.diff(z[i-w:i+1], axis=0)
                cti[i] = np.linalg.norm(np.var(v, axis=0))
            return cti

        for sp_name, imgs, mask, pv, times in [
            ("train", sf_train_imgs[train_mask], None, sf_train_pv[train_mask], ts_tv[train_mask]),
            ("val",   sf_train_imgs[val_mask],   None, sf_train_pv[val_mask],   ts_tv[val_mask]),
            ("test",  sf_test_imgs,              None, sf_test_pv,               pd.to_datetime(sf_test_times)),
        ]:
            print(f"  encoding {sp_name} ({len(imgs)} samples)")
            z = encode_imgs(imgs)
            cti = cti_window(z, w=10)
            np.save(SF_LATENTS_DIR / f"{sp_name}_latents.npy", z)
            np.save(SF_LATENTS_DIR / f"{sp_name}_cti.npy", cti)
            np.save(SF_LATENTS_DIR / f"{sp_name}_pv.npy", pv.astype(np.float32))
            # Stanford has no GHI, so use PV/PV_scale as the equivalent of "kt"
            kt_proxy = (pv / SF_PV_SCALE).astype(np.float32)
            np.save(SF_LATENTS_DIR / f"{sp_name}_kt.npy", kt_proxy)
            # Minimal covariates: hour_sin/cos, doy_sin/cos, kt itself (autoregressive)
            ts = times
            hf = (ts.hour + ts.minute / 60.0).values
            doy = ts.dayofyear.values
            cov = np.stack([
                np.sin(2*np.pi*hf/24).astype(np.float32),
                np.cos(2*np.pi*hf/24).astype(np.float32),
                np.sin(2*np.pi*doy/365.25).astype(np.float32),
                np.cos(2*np.pi*doy/365.25).astype(np.float32),
                kt_proxy,
            ], axis=1)
            np.save(SF_LATENTS_DIR / f"{sp_name}_covariates.npy", cov)
        del vae; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- A.5 Train Stanford SDE + Score Decoder ----
    if not SF_SCORE_CKPT.exists():
        print("[A.5] Training Stanford SDE + Score Decoder ...")
        sf_z_tr = np.load(SF_LATENTS_DIR / "train_latents.npy")
        sf_cti_tr = np.load(SF_LATENTS_DIR / "train_cti.npy")
        sf_kt_tr = np.load(SF_LATENTS_DIR / "train_kt.npy")
        sf_cov_tr = np.load(SF_LATENTS_DIR / "train_covariates.npy")
        sf_z_val = np.load(SF_LATENTS_DIR / "val_latents.npy")
        sf_cti_val = np.load(SF_LATENTS_DIR / "val_cti.npy")
        sf_kt_val = np.load(SF_LATENTS_DIR / "val_kt.npy")
        sf_cov_val = np.load(SF_LATENTS_DIR / "val_covariates.npy")

        # SDE — same architecture, dt=60s for Stanford (1-min sampling)
        sf_sde = LatentNeuralSDE(z_dim=64, c_dim=sf_cov_tr.shape[1]).to(DEVICE)
        opt_sde = torch.optim.Adam(sf_sde.parameters(), lr=5e-4)

        # Mixed-horizon dataset
        class SfMHDS(Dataset):
            def __init__(self, z, cti, c, hs=(1, 5, 10, 15, 30), seed=42):
                self.z = z; self.cti = cti; self.c = c; self.hs = hs
                self.rng = np.random.RandomState(seed)
                self.maxh = max(hs)
                self.idx = np.arange(len(z) - self.maxh)
            def __len__(self): return len(self.idx)
            def __getitem__(self, i):
                ii = self.idx[i]; k = int(self.rng.choice(self.hs))
                return {
                    "z_t": torch.from_numpy(self.z[ii]),
                    "z_next": torch.from_numpy(self.z[ii + k]),
                    "k": torch.tensor(k, dtype=torch.float32),
                    "cti_t": torch.tensor(self.cti[ii], dtype=torch.float32),
                    "c_t": torch.from_numpy(self.c[ii]),
                }
        ds = SfMHDS(sf_z_tr, sf_cti_tr, sf_cov_tr)
        dl = DataLoader(ds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

        print(f"  SDE: training on {len(ds)} mixed-horizon transitions")
        for ep in range(1, 31):
            sf_sde.train(); tl_d = tl_s = 0; n = 0
            for b in dl:
                z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
                k = b["k"].float().unsqueeze(-1).to(DEVICE)
                t = (k / 30.0)
                cti = b["cti_t"].unsqueeze(-1).to(DEVICE)
                c = b["c_t"].to(DEVICE)
                mu = sf_sde.drift(z, t, c)
                sigma = sf_sde.diffusion(z, cti)
                dz = (zn - z) / k
                drift_loss = F.mse_loss(mu, dz)
                resid = zn - z - mu * k
                target_var = (resid ** 2) / k.clamp(min=1.0)
                sigma_sq = sigma.pow(2).clamp(min=1e-6)
                diff_loss = F.mse_loss(torch.log(sigma_sq + 1e-8), torch.log(target_var + 1e-8))
                loss = drift_loss + 0.5 * diff_loss
                opt_sde.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(sf_sde.parameters(), 1.0); opt_sde.step()
                tl_d += drift_loss.item(); tl_s += diff_loss.item(); n += 1
            print(f"  SDE ep {ep}/30: drift={tl_d/n:.4f} diff={tl_s/n:.4f}")
        torch.save(sf_sde.state_dict(), SF_SDE_CKPT)

        # Score Decoder for Stanford (predict delta-kt over kt-proxy = PV/PV_scale)
        sf_score = CondScoreDecoder(z_dim=64, c_dim=sf_cov_tr.shape[1], predict_mode='delta').to(DEVICE)
        opt_score = torch.optim.Adam(sf_score.parameters(), lr=1e-4)

        class SfScoreDS(Dataset):
            def __init__(self, z, cti, c, kt, hs=(1, 5, 10, 15, 30), seed=42):
                self.z = z; self.cti = cti; self.c = c; self.kt = kt; self.hs = hs
                self.rng = np.random.RandomState(seed)
                self.maxh = max(hs)
            def __len__(self): return len(self.z) - self.maxh
            def __getitem__(self, i):
                k = int(self.rng.choice(self.hs))
                return {
                    "kt_target": torch.tensor(self.kt[i + k], dtype=torch.float32),
                    "kt_current": torch.tensor(self.kt[i], dtype=torch.float32),
                    "z_t": torch.from_numpy(self.z[i]),
                    "cti_t": torch.tensor(self.cti[i], dtype=torch.float32),
                    "c_t": torch.from_numpy(self.c[i]),
                }
        score_ds = SfScoreDS(sf_z_tr, sf_cti_tr, sf_cov_tr, sf_kt_tr)
        score_dl = DataLoader(score_ds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

        for ep in range(1, 31):
            sf_score.train(); tl = 0; n = 0
            for b in score_dl:
                kt_t = b["kt_target"].unsqueeze(-1).to(DEVICE)
                kt_c = b["kt_current"].unsqueeze(-1).to(DEVICE)
                z = b["z_t"].to(DEVICE); cti = b["cti_t"].unsqueeze(-1).to(DEVICE)
                c = b["c_t"].to(DEVICE)
                loss = sf_score.training_loss(kt_t, kt_c, z, cti, c)
                opt_score.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(sf_score.parameters(), 1.0); opt_score.step()
                tl += loss.item(); n += 1
            print(f"  Score ep {ep}/30: loss={tl/n:.4f}")
        torch.save(sf_score.state_dict(), SF_SCORE_CKPT)
        del sf_sde, sf_score; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    else:
        print("[A.5] Stanford SDE + Score checkpoints exist, skipping.")

    # ---- A.6 Evaluate Stanford SolarSDE on test set ----
    print("[A.6] Evaluating Stanford SolarSDE on test set ...")
    sf_z_te = np.load(SF_LATENTS_DIR / "test_latents.npy")
    sf_cti_te = np.load(SF_LATENTS_DIR / "test_cti.npy")
    sf_pv_te = np.load(SF_LATENTS_DIR / "test_pv.npy")
    sf_kt_te = np.load(SF_LATENTS_DIR / "test_kt.npy")
    sf_cov_te = np.load(SF_LATENTS_DIR / "test_covariates.npy")

    sf_sde = LatentNeuralSDE(z_dim=64, c_dim=sf_cov_te.shape[1]).to(DEVICE)
    sf_sde.load_state_dict(torch.load(SF_SDE_CKPT, map_location=DEVICE, weights_only=False))
    sf_sde.eval()
    sf_score = CondScoreDecoder(z_dim=64, c_dim=sf_cov_te.shape[1], predict_mode='delta').to(DEVICE)
    sf_score.load_state_dict(torch.load(SF_SCORE_CKPT, map_location=DEVICE, weights_only=False))
    sf_score.eval()

    HORIZONS_SF = [1, 5, 10, 15, 30]   # minutes (= timesteps for 1-min Stanford)
    N_SAMPLES = 50
    rows = []
    with torch.no_grad():
        for h in HORIZONS_SF:
            preds_all = []
            truths = []
            for i in tqdm(range(0, len(sf_z_te) - h, 4), desc=f"h={h}"):
                z0 = torch.from_numpy(sf_z_te[i]).unsqueeze(0).repeat(N_SAMPLES, 1).to(DEVICE)
                cti0 = torch.tensor(sf_cti_te[i]).unsqueeze(0).unsqueeze(-1).repeat(N_SAMPLES, 1).to(DEVICE)
                c0 = torch.from_numpy(sf_cov_te[i]).unsqueeze(0).repeat(N_SAMPLES, 1).to(DEVICE)
                kt0 = torch.tensor(sf_kt_te[i]).unsqueeze(0).unsqueeze(-1).repeat(N_SAMPLES, 1).to(DEVICE)
                # Roll latent forward h steps via Euler-Maruyama
                z = z0
                for s in range(h):
                    t_norm = torch.full((N_SAMPLES, 1), float(s) / 30.0, device=DEVICE)
                    mu = sf_sde.drift(z, t_norm, c0)
                    sigma = sf_sde.diffusion(z, cti0)
                    z = z + mu * 1.0 + sigma * torch.randn_like(z)
                # Decode kt at horizon
                kt_pred = sf_score.sample(z, cti0, c0, kt0, n=1).squeeze(-1).cpu().numpy()
                pv_pred = kt_pred * SF_PV_SCALE
                preds_all.append(pv_pred)
                truths.append(sf_pv_te[i + h])
            preds = np.array(preds_all)         # (N_obs, N_samples)
            tru = np.array(truths)              # (N_obs,)
            crps = crps_empirical(tru, preds).mean()
            rmse = np.sqrt(((preds.mean(1) - tru) ** 2).mean())
            picp = ((np.percentile(preds, 5, axis=1) <= tru) &
                    (tru <= np.percentile(preds, 95, axis=1))).mean()
            pinaw = (np.percentile(preds, 95, axis=1) - np.percentile(preds, 5, axis=1)).mean() / max(tru.max() - tru.min(), 1.0)
            print(f"  h={h}: CRPS={crps:.3f}  RMSE={rmse:.3f}  PICP={picp:.3f}  PINAW={pinaw:.3f}")
            rows.append({"horizon_min": h, "crps": crps, "rmse": rmse, "picp": picp, "pinaw": pinaw})
            np.save(SF_RESULTS_DIR / f"solarsde_preds_h{h}.npy", preds)
            np.save(SF_RESULTS_DIR / f"truths_h{h}.npy", tru)
    pd.DataFrame(rows).to_csv(SF_RESULTS_DIR / "solarsde_results.csv", index=False)
    print(f"\nStanford SolarSDE results -> {SF_RESULTS_DIR / 'solarsde_results.csv'}")
    del sf_sde, sf_score; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## Final: zip Part 1 outputs

In [ ]:
# ==== Zip and download all outputs ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs_combined.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} ...")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive: {zip_path}  ({size_mb:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    try: files.download(str(zip_path))
    except Exception as e: print(f"Auto-download failed: {e}. File at {zip_path}")
elif IN_KAGGLE:
    # Copy the zip + the two headline CSVs to /kaggle/working/ top-level
    # so they show up prominently in the Output tab of the notebook.
    top = Path("/kaggle/working")
    shutil.copy(zip_path, top / zip_path.name)
    for key_file in ["results/main_results_combined.csv",
                     "results/ablation_results.csv",
                     "results/solar_sde_calibrated.csv"]:
        src = PERSIST_DIR / key_file
        if src.exists():
            shutil.copy(src, top / src.name)
    print(f"Kaggle: zip + key CSVs copied to /kaggle/working/. Use Output tab to download,")
    print(f"or 'Save Version' to commit them permanently as notebook outputs.")
else:
    print(f"Local: file at {zip_path}")

print("\n" + "=" * 70)
print("ALL STAGES COMPLETE")
print("=" * 70)
for sub in ["splits", "extended", "checkpoints", "latents", "results", "figures"]:
    p = PERSIST_DIR / sub
    if p.exists():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        print(f"  {sub}/: {n} files, {total/1e6:.1f} MB")
